# Reducing Production-Line Sensor Redundancy with PROC GVARCLUS

## Executive Summary

A multi-zone manufacturing line streams dozens of correlated sensor readings, many of which carry the same underlying signal. This notebook uses **PROC GVARCLUS** (variable clustering) to group the 13 process sensors into four disjoint clusters, then reads each cluster's R-squared values to choose one representative gauge per cluster — cutting the SPC monitoring footprint without losing process information. Three of the four clusters map cleanly onto physical zones (average R-squared 0.92, 0.93, and 0.96); the fourth is a single-channel cluster the procedure split off, which we examine rather than gloss over.

## Data Sources

All data is generated inline with `call streaminit(20260531)` and `rand()` — no external or network inputs.

| Dataset | Rows | Key Variables | Description |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Synthetic readings from a 3-zone production line. Sensors within a zone share a latent driver (high correlation); cross-zone gauges (temperatures tied to zone 1, pressures to zone 2, vibration to zone 3) are added to create realistic redundancy. The DATA step loop requests 300 cycles, but this evaluation environment runs in unlicensed mode and retains the first 100 observations — enough to recover the cluster structure cleanly. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | One row per input sensor: the cluster it was assigned to and its R-squared with that cluster's component. Produced by the `OUTPUT OUT=` statement. |

# Reducing Production-Line Sensor Redundancy with PROC GVARCLUS

On an instrumented production line, it is common to log dozens of sensors that measure overlapping physical phenomena — multiple thermocouples in one zone, redundant pressure transducers, vibration probes that track the same shaft. Feeding every channel into a control chart or downstream model wastes monitoring effort and inflates multicollinearity.

**Variable clustering** addresses this directly. `PROC GVARCLUS` partitions the numeric variables into *disjoint* clusters, where each cluster is summarized by the first principal component of its members. Sensors that move together land in the same cluster; the engineer can then keep one representative channel per cluster.

In this notebook we:

1. Synthesize readings from a 3-zone line with deliberately redundant sensors (the loop requests 300 cycles; 100 are retained here).
2. Cluster the 13 channels with `PROC GVARCLUS` at `MAXCLUSTERS=4`.
3. Capture the cluster assignments in an output dataset and summarize them.
4. Interpret the R-squared values to pick one gauge per cluster for ongoing SPC.

## Step 1 — Generate synthetic sensor data

We simulate three process zones, each with a hidden driver (`zone1_a`, `zone2_a`, `zone3_a`). The remaining channels are noisy linear functions of their zone's driver, so they are strongly correlated within a zone but nearly independent across zones. We also tie the inlet/outlet temperatures to zone 1 and the two pressure transducers to zone 2, mimicking the cross-instrument redundancy seen on real lines. A fixed seed makes the run reproducible. The loop requests 300 cycles; in unlicensed mode the engine retains the first 100 observations, which the NOTE below reports.

In [1]:
data process_sensors;
    call streaminit(20260531);
    do cycle = 1 to 300;
        /* Zone 1: latent driver plus two redundant probes */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Zone 2: latent driver plus two redundant probes */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Zone 3: latent driver plus one redundant probe */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Cross-instrument channels tied to existing drivers */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        output;
    end;
    drop cycle;
run;

NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Step 2 — Cluster the sensors

We call `PROC GVARCLUS` on all 13 channels. The `VAR` statement lists the numeric sensors to cluster. We cap the partition at four clusters with `MAXCLUSTERS=4` (we expect roughly one cluster per physical zone, with a little slack). `ODS GRAPHICS ON` with `PLOTS=ALL` enables the variable-cluster dendrogram; the engine writes that SVG to the working directory rather than rendering it inline, so the structure we read below comes from the printed **Variable Cluster Assignments** table and the per-cluster eigenvalue table.

The `OUTPUT OUT=` statement writes the variable-to-cluster assignments — along with each variable's R-squared against its own cluster — to a dataset we can program against later.

In [2]:
ods graphics on;

proc gvarclus data=process_sensors maxclusters=4 plots=all;
    var zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    output out=clusters;
run;

ods graphics off;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  ZONE1_A                                 1     0.968670
  ZONE1_B                                 1     0.918900
  ZONE1_C                                 4     1.000000
  ZONE2_A                                 2     0.983640
  ZONE2_B                                 2     0.946708
  ZONE2_C                                 2     0.892405
  ZONE3_A                                 3     0.977237
  ZONE3_B                                 3     0.949054
  TEMP_IN                                 1     0.903394
  TEMP_OUT                                1     0.889519
  PRESSURE_A                              2     0.929356
  PRESSURE_B                              2     0.920915
  VIBRATION  

## Step 3 — Re-run with the SUMMARY option

Running the procedure a second time with the `SUMMARY` option keeps the same partition. `PLOTS=NONE` turns graphics off on this pass. In this environment the printed report is identical to Step 2 — the same 13-row assignment table, the same four clusters, and the same eigenvalue summary — so this cell mainly demonstrates that the `SUMMARY` and `PLOTS=NONE` options parse and run; it is not expected to add new numbers.

In [3]:
proc gvarclus data=process_sensors maxclusters=4 summary plots=none;
    var zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
run;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  ZONE1_A                                 1     0.968670
  ZONE1_B                                 1     0.918900
  ZONE1_C                                 4     1.000000
  ZONE2_A                                 2     0.983640
  ZONE2_B                                 2     0.946708
  ZONE2_C                                 2     0.892405
  ZONE3_A                                 3     0.977237
  ZONE3_B                                 3     0.949054
  TEMP_IN                                 1     0.903394
  TEMP_OUT                                1     0.889519
  PRESSURE_A                              2     0.929356
  PRESSURE_B                              2     0.920915
  VIBRATION  

## Step 4 — Inspect the output dataset

The `clusters` dataset from `OUTPUT OUT=` has one row per sensor with its assigned `Cluster` and `RSq_Own` (the squared correlation between the sensor and its cluster component). A high `RSq_Own` means the sensor is well represented by its cluster — an ideal candidate to *drop* in favor of the cluster's representative. We sort by cluster, then by R-squared, so the most representative sensor in each cluster sits at the top of its group.

In [4]:
proc sort data=clusters out=clusters_ranked;
    by Cluster descending RSq_Own;
run;

proc print data=clusters_ranked label;
    var Variable Cluster RSq_Own;
    label Variable = "Sensor Channel"
          Cluster  = "Cluster"
          RSq_Own  = "R-Square with Own Cluster";
run;


  Obs  Sensor Channel  Cluster  R-Square with Own Cluster
-----  --------------  -------  -------------------------
    1  ZONE1_A               1                    0.96867
    2  ZONE1_B               1                     0.9189
    3  TEMP_IN               1                   0.903394
    4  TEMP_OUT              1                   0.889519
    5  ZONE2_A               2                    0.98364
    6  ZONE2_B               2                   0.946708
    7  PRESSURE_A            2                   0.929356
    8  PRESSURE_B            2                   0.920915
    9  ZONE2_C               2                   0.892405
   10  ZONE3_A               3                   0.977237
   11  VIBRATION             3                    0.95916
   12  ZONE3_B               3                   0.949054
   13  ZONE1_C               4                          1

NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE

## Interpreting the results

The partition recovers most of the physical structure of the line, with one honest caveat:

- **Cluster 1** gathers `zone1_a` (R²=0.969), `zone1_b` (0.919) and the inlet/outlet temperatures `temp_in` (0.903) and `temp_out` (0.890) — four of the five channels driven by the zone-1 latent signal. Average R-squared 0.920.
- **Cluster 2** gathers all three zone-2 channels `zone2_a` (0.984), `zone2_b` (0.947), `zone2_c` (0.892) together with the two pressure transducers `pressure_a` (0.929) and `pressure_b` (0.921), which we tied to the zone-2 driver. Average R-squared 0.935.
- **Cluster 3** gathers `zone3_a` (0.977), `zone3_b` (0.949) and `vibration` (0.959). Average R-squared 0.962.
- **Cluster 4** is a single channel: `zone1_c`, the third zone-1 probe, was split off on its own with R²=1.000 (a singleton is, trivially, perfectly explained by itself). Despite sharing the zone-1 driver, at this sample size the procedure judged `zone1_c` distinct enough from the `zone1_a`/temperature group to warrant its own cluster rather than merging it into Cluster 1.

The three multi-channel clusters each carry an average R-squared above **0.90**, confirming that a single component explains the bulk of the variation among their members — exactly the redundancy we want to collapse. The lone outlier — `zone1_c` landing in its own cluster instead of with the rest of zone 1 — is a useful reminder that variable clustering is data-driven: with more cycles or a higher `MAXCLUSTERS` budget the boundary can shift, so the partition is a starting point for engineering judgment, not a fixed truth.

**Action for the line.** Within each multi-channel cluster, keep the sensor with the highest `RSq_Own` (the channel most representative of its cluster) and retire or de-prioritize the rest from routine SPC charting — for example `zone2_a` for Cluster 2 and `zone3_a` for Cluster 3. Treat the `zone1_c` singleton as a flag to investigate rather than an automatic keep: confirm whether it carries genuinely distinct information or is a clustering artifact before deciding to monitor it separately. Even with that one channel held aside, this collapses 13 monitored channels toward a four-channel monitoring plan, cutting alarm noise and multicollinearity while preserving the information content of the line.